<a href="https://colab.research.google.com/github/Yuuraajaa/Tito_Dwi_Saputro_2411007-Metode-Ai/blob/main/Tito_Dwi_Saputro_2411007_P2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 1. INSTALL & IMPORT
# ============================================================

!pip install -q kaggle

import os
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from google.colab import files
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG19
from tensorflow.keras.applications.vgg19 import preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.21.0
GPU: []


In [ ]:
# ============================================================
# 2. DOWNLOAD DATASET KAGGLE
# ============================================================

# Upload kaggle.json dari akun Kaggle
files.upload()

Saving cats_dogs_light.zip to cats_dogs_light.zip
Buffered data was truncated after reaching the output size limit.

In [ ]:
# Buat konfigurasi Kaggle

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [ ]:
mkdir -p ~/.kaggle && echo KGAT_4da6e750782fa487b6b1ccb2a3ee7369 > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

In [ ]:
# Download dataset Dogs vs Cats
!kaggle competitions download -c dogs-vs-cats

403 Client Error: Forbidden for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/DownloadDataFiles


In [ ]:
# Extract

!unzip -q dogs-vs-cats.zip
!unzip -q train.zip

unzip:  cannot find or open dogs-vs-cats.zip, dogs-vs-cats.zip.zip or dogs-vs-cats.zip.ZIP.
unzip:  cannot find or open train.zip, train.zip.zip or train.zip.ZIP.


In [ ]:
# Cek dataset

print("Jumlah gambar:", len(os.listdir("/content/train")))

In [ ]:
# ============================================================
# 3. BUAT DATASET TRAIN & VALIDATION
# ============================================================

SOURCE_DIR = "/content/train"
BASE_DIR = "/content/dogs_vs_cats"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR = os.path.join(BASE_DIR, "validation")

for folder in [
    os.path.join(TRAIN_DIR, "cats"),
    os.path.join(TRAIN_DIR, "dogs"),
    os.path.join(VAL_DIR, "cats"),
    os.path.join(VAL_DIR, "dogs")
]:
    os.makedirs(folder, exist_ok=True)

In [ ]:
# Ambil 10.000 gambar agar training lebih cepat

files_list = [
    f for f in os.listdir(SOURCE_DIR)
    if f.endswith(".jpg")
]

random.seed(42)
random.shuffle(files_list)

files_list = files_list[:10000]

train_files = files_list[:8000]
val_files = files_list[8000:]

print("Training   :", len(train_files))
print("Validation :", len(val_files))

In [ ]:
# Copy training images

for filename in train_files:

    if filename.startswith("cat"):
        label = "cats"
    else:
        label = "dogs"

    shutil.copy(
        os.path.join(SOURCE_DIR, filename),
        os.path.join(TRAIN_DIR, label, filename)
    )

In [ ]:
# Copy validation images

for filename in val_files:

    if filename.startswith("cat"):
        label = "cats"
    else:
        label = "dogs"

    shutil.copy(
        os.path.join(SOURCE_DIR, filename),
        os.path.join(VAL_DIR, label, filename)
    )

In [ ]:
# ============================================================
# 4. LOAD DATASET
# ============================================================

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

print(train_ds.class_names)

In [ ]:
# ============================================================
# 5. PREPROCESSING VGG19
# ============================================================

def preprocess_vgg19(image, label):
    image = preprocess_input(image)
    return image, label


train_ds = train_ds.map(
    preprocess_vgg19,
    num_parallel_calls=tf.data.AUTOTUNE
)

val_ds = val_ds.map(
    preprocess_vgg19,
    num_parallel_calls=tf.data.AUTOTUNE
)

train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

In [ ]:
# ============================================================
# 6. LOAD VGG19 PRETRAINED IMAGENET
# ============================================================

base_model = VGG19(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.summary()

In [ ]:
# ============================================================
# 7. FREEZE VGG19
# ============================================================

base_model.trainable = False

print("Trainable:", base_model.trainable)

In [ ]:
# ============================================================
# 8. BUAT MODEL TRANSFER LEARNING
# ============================================================

model = models.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(
        128,
        activation="relu"
    ),

    layers.Dropout(0.5),

    layers.Dense(
        1,
        activation="sigmoid"
    )
])

model.summary()

In [ ]:
# ============================================================
# 9. COMPILE
# ============================================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),

    loss="binary_crossentropy",

    metrics=["accuracy"]
)

In [ ]:
# ============================================================
# 10. TRAINING TRANSFER LEARNING
# ============================================================

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

In [ ]:
# ============================================================
# 11. PLOT ACCURACY
# ============================================================

plt.figure(figsize=(8,5))

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.legend()
plt.grid()

plt.show()

In [ ]:
# ============================================================
# 12. PLOT LOSS
# ============================================================

plt.figure(figsize=(8,5))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.legend()
plt.grid()

plt.show()

In [ ]:
# ============================================================
# 13. EVALUATION
# ============================================================

loss, accuracy = model.evaluate(val_ds)

print("Validation Loss     :", loss)
print("Validation Accuracy :", accuracy)

In [ ]:
# ============================================================
# 14. PREDIKSI GAMBAR
# ============================================================

def predict_image(image_path):

    img = load_img(
        image_path,
        target_size=(224, 224)
    )

    img_array = img_to_array(img)

    img_array = np.expand_dims(
        img_array,
        axis=0
    )

    img_array = preprocess_input(img_array)

    prediction = model.predict(img_array, verbose=0)[0][0]

    if prediction >= 0.5:
        label = "DOG"
        probability = prediction
    else:
        label = "CAT"
        probability = 1 - prediction

    plt.figure(figsize=(5,5))
    plt.imshow(load_img(image_path))
    plt.axis("off")
    plt.title(
        f"{label} ({probability*100:.2f}%)"
    )
    plt.show()


predict_image(
    "/content/dogs_vs_cats/validation/dogs/" +
    os.listdir("/content/dogs_vs_cats/validation/dogs")[0]
)

Pertanyaan analisis
1. Apa perbedaan training dari scratch dan transfer learning?
2. Mengapa pretrained model dapat belajar lebih cepat?
3. Apa fungsi freezing layer?
4. Apa perbedaan feature extraction dan fine-tuning?
5. Apakah fine-tuning selalu meningkatkan accuracy?
6. Bagaimana pengaruh jumlah data terhadap pemilihan strategi?

**Jawaban**
1. Apa perbedaan training from scratch dan transfer learning?
Training from scratch adalah proses melatih model dari awal dengan bobot yang diinisialisasi secara acak sehingga model harus mempelajari fitur dari data yang diberikan. Sementara itu, transfer learning menggunakan model yang sudah dilatih sebelumnya (pretrained model) untuk menyelesaikan tugas baru. Transfer learning biasanya membutuhkan waktu pelatihan dan data yang lebih sedikit.
2. Mengapa pretrained model dapat belajar lebih cepat?
Karena pretrained model sudah memiliki pengetahuan berupa fitur yang dipelajari dari dataset sebelumnya. Misalnya, pada pengenalan gambar, model sudah dapat mengenali pola dasar seperti garis, bentuk, dan tekstur. Jadi, model tidak perlu mempelajari semuanya dari awal dan bisa lebih fokus pada karakteristik objek yang baru.
3. Apa fungsi freezing layer?
Freezing layer berfungsi untuk mempertahankan bobot pada lapisan tertentu agar tidak berubah selama proses training. Tujuannya adalah menjaga fitur yang sudah dipelajari sebelumnya, mengurangi jumlah parameter yang dilatih, serta mempercepat proses training dan mengurangi risiko overfitting ketika data terbatas.
4. Apa perbedaan feature extraction dan fine-tuning?
Feature extraction menggunakan pretrained model sebagai pengambil fitur dengan membekukan lapisan model tersebut, kemudian melatih lapisan baru untuk tugas yang diinginkan. Sementara itu, fine-tuning melibatkan pelatihan ulang sebagian atau seluruh lapisan pretrained model agar bobotnya dapat menyesuaikan dengan dataset baru.
5. Apakah fine-tuning selalu meningkatkan accuracy?
Tidak selalu. Fine-tuning dapat meningkatkan akurasi jika model berhasil menyesuaikan fitur dengan data baru. Namun, jika data terlalu sedikit, learning rate terlalu besar, atau data baru berbeda jauh dari data awal, fine-tuning justru dapat menyebabkan overfitting atau menurunkan akurasi model.
6. Bagaimana pengaruh jumlah data terhadap pemilihan strategi?
Jumlah data memengaruhi strategi pelatihan yang dipilih. Jika data sedikit, transfer learning dengan feature extraction biasanya lebih cocok karena memanfaatkan fitur yang sudah dipelajari dan mengurangi risiko overfitting. Jika data cukup banyak dan relevan, fine-tuning dapat digunakan agar model lebih sesuai dengan tugas baru. Sementara itu, training from scratch umumnya membutuhkan data yang lebih banyak agar model dapat belajar dengan baik.